# Sprint 5

## Install PySpark

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    print("Installed pyspark.")
else:
    print("pyspark is already available.")

## Spark session

In [ ]:
# Windows warning

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CS131_bladdards")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

## Import the funtions and create data path

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve # to download data if not already present

from pyspark.sql.functions import (
    col,
    to_date,
    avg,
    count,
    broadcast
)

# Path for MIMIC-IV data
DATA_DIR = Path("../data/MIMIC-IV/hosp")


## Dataframes

In [ ]:
# -------------------------------------------------------
# Visits dataframe
# -------------------------------------------------------


In [ ]:
# -------------------------------------------------------
# BHOOMIKA'S SECTION — diagnoses DataFrame
# Collects all diagnoses for visits of interest
# (pre-BC symptom visits + first BC diagnosis visits)
# -------------------------------------------------------

EVIDENCE_DIR = Path("../out/evidence")
# DATA_DIR is already defined above as Path("data/MIMIC-IV/hosp")

# STEP 1: Load visits of interest from pre_bc_symptom_timeline
# row_type (SYMPTOM or BC_FIRST_DX) becomes visit_type
timeline_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(EVIDENCE_DIR / "pre_bc_symptom_timeline.csv"))
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("row_type").alias("visit_type")
    )
    .dropDuplicates(["hadm_id"])  # one visit_type label per admission
)

print("Timeline visits of interest:", timeline_df.count())
timeline_df.show(5)

# STEP 2: Load all diagnoses from MIMIC
# seq_num = order diagnoses were recorded per visit → becomes ranking
dx_icd_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "diagnoses_icd.csv.gz"))
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("seq_num").cast("int").alias("ranking"),
        col("icd_code"),
        col("icd_version").cast("int")
    )
)

print("diagnoses_icd rows:", dx_icd_df.count())
dx_icd_df.show(5)

# STEP 3: Load ICD code dictionary
# Maps icd_code + icd_version → human readable description
icd_dict_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "d_icd_diagnoses.csv.gz"))
    .select(
        col("icd_code"),
        col("icd_version").cast("int"),
        col("long_title").alias("icd_desc")
    )
)

print("ICD dictionary rows:", icd_dict_df.count())
icd_dict_df.show(5)

# STEP 4: Filter diagnoses to visits of interest only
# Inner join on hadm_id — keeps only admissions in our timeline
# broadcast(timeline_df) since it is small (1568 rows vs 6M+)
filtered_dx_df = dx_icd_df.join(
    broadcast(timeline_df),
    on="hadm_id",
    how="inner"
)

print("Diagnoses for visits of interest:", filtered_dx_df.count())

# Drop duplicate subject_id introduced by the join
# (both dx_icd_df and timeline_df have subject_id)
filtered_dx_df2 = filtered_dx_df.drop(timeline_df["subject_id"])

# STEP 5: Enrich with ICD descriptions
# Left join on icd_code + icd_version — must match both since
# same code can mean different things in ICD-9 vs ICD-10
diagnoses = (
    filtered_dx_df2.join(
        broadcast(icd_dict_df),  # dictionary is small, broadcast it
        on=["icd_code", "icd_version"],
        how="left"  # keep all rows even if no dictionary entry found
    )
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("visit_type").cast("string"),
        col("ranking").cast("int"),
        col("icd_code").cast("string"),
        col("icd_version").cast("int"),
        col("icd_desc").cast("string")
    )
    .orderBy("subject_id", "hadm_id", "ranking")
)

print("=== diagnoses DataFrame ===")
print("Row count:", diagnoses.count())
diagnoses.printSchema()
diagnoses.show(10, truncate=False)


In [ ]:
# -------------------------------------------------------
# BHOOMIKA'S SECTION for icd_codes lookup table
# Goal: all ICD codes with a status label
# NOT_RELATED = general code
# RELEVANT = symptom related to BC (from symptom_icd_list.txt)
# BC_DIAGNOSIS = confirmed BC code (from bc_icd_codes.csv)
# -------------------------------------------------------

# Step 1: Load full ICD dictionary as base
icd_codes = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "d_icd_diagnoses.csv.gz"))
    .select(
        col("icd_code").cast("string"),
        col("icd_version").cast("int"),
        col("long_title").alias("description")
    )
)

print("Total ICD codes:", icd_codes.count())
icd_codes.show(5)

# Step 2: Load BC diagnosis codes from sprint 2 output
bc_codes_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("../data/bc_icd_codes.csv")
    .select(
        col("icd_code").cast("string"),
        col("icd_version").cast("int")
    )
)

print("BC diagnosis codes:", bc_codes_df.count())
bc_codes_df.show(5)

# Step 3: Load symptom ICD codes from sprint 3 output
symptom_codes_df = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "true")
    .csv("../data/symptom_icd_list.txt")
    .toDF("icd_code", "icd_version")
    .filter(col("icd_code") != "icd_code")
    .select(
        col("icd_code").cast("string"),
        col("icd_version").cast("int")
    )
)

print("Symptom codes:", symptom_codes_df.count())
symptom_codes_df.show(5)

# Step 4: Left join BC codes onto full ICD list
# flag = 1 if it's a BC diagnosis code
icd_with_bc = icd_codes.join(
    broadcast(bc_codes_df.withColumn("is_bc", col("icd_code").isNotNull())),
    on=["icd_code", "icd_version"],
    how="left"
)

# Step 5: Left join symptom codes
# flag = 1 if it's a relevant symptom code
icd_with_both = icd_with_bc.join(
    broadcast(symptom_codes_df.withColumn("is_symptom", col("icd_code").isNotNull())),
    on=["icd_code", "icd_version"],
    how="left"
)

# Step 6: Derive status column
# BC_DIAGNOSIS takes priority over RELEVANT
from pyspark.sql.functions import when

icd_codes = (
    icd_with_both
    .select(
        col("icd_code").cast("string"),
        col("icd_version").cast("int"),
        col("description").cast("string"),
        when(col("is_bc") == True, "BC_DIAGNOSIS")
        .when(col("is_symptom") == True, "RELEVANT")
        .otherwise("NOT_RELATED")
        .alias("status")
    )
    .orderBy("icd_code", "icd_version")
)

print("=== icd_codes lookup table ===")
print("Row count:", icd_codes.count())
icd_codes.printSchema()
icd_codes.show(10, truncate=False)

# Quick check — how many of each status?
print("Status distribution:")
icd_codes.groupBy("status").agg(count("*").alias("count")).show()

In [ ]:
# -------------------------------------------------------
# Aggregation
# -------------------------------------------------------


# -------------------------------------------------------
# BHOOMIKA'S SECTION — Symptom Diagnosis Rankings
# Goal: Check how relevant symptom visits are by finding
# the minimum ranking of relevant ICD codes per visit
# -------------------------------------------------------

from pyspark.sql.functions import min as spark_min, avg as spark_avg, count

# Filter diagnoses to symptom visits only
symptom_visits_df = diagnoses.filter(col("visit_type") == "SYMPTOM")

# Get only RELEVANT codes from our icd_codes lookup table
relevant_icd_df = icd_codes.filter(col("status") == "RELEVANT")

# Inner join — keep only symptom rows where icd_code is RELEVANT
relevant_symptoms_df = symptom_visits_df.join(
    broadcast(relevant_icd_df),
    on=["icd_code", "icd_version"],
    how="inner"
)

print("Relevant symptom rows:", relevant_symptoms_df.count())
relevant_symptoms_df.show(5)

# Group by visit (hadm_id) and find minimum ranking per visit
# Lower ranking = more important diagnosis
symptom_rankings = (
    relevant_symptoms_df
    .groupBy("hadm_id")
    .agg(
        spark_min(col("ranking")).alias("symptom_rankings")
    )
)

print("=== Symptom Rankings Summary ===")
symptom_rankings.select("symptom_rankings").summary().show()


## Clean up
Stop spark session when done

In [ ]:
# Uncomment when you are completely done:

# spark.stop()